# Machine Learning: A Leakage-Aware Regression Pipeline

**Type:** Original supplementary practice  
**Data:** Synthetic tabular dataset generated in the notebook

The goal is to demonstrate a reproducible scikit-learn pipeline with numeric and categorical preprocessing, a held-out test set, cross-validation and permutation importance. The simulated target has no real-world interpretation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(71)
n = 1600
df = pd.DataFrame({
    "floor_area": rng.normal(105, 28, n).clip(25, 260),
    "rooms": rng.integers(1, 7, n),
    "building_age": rng.integers(0, 45, n),
    "district": rng.choice(["Central","North","South","East"], n),
    "property_type": rng.choice(["Apartment","House","Duplex"], n, p=[.65,.25,.10])
})
district_effect = df["district"].map({"Central":35,"North":12,"South":-8,"East":5})
type_effect = df["property_type"].map({"Apartment":0,"House":22,"Duplex":30})
noise = rng.normal(0, 18, n)
y = 45 + .75*df["floor_area"] + 8*df["rooms"] - .65*df["building_age"] + district_effect + type_effect + noise
# Add small amounts of missing feature data to exercise preprocessing.
for col in ["floor_area","building_age","district"]:
    df.loc[rng.choice(df.index, 25, replace=False), col] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=.2, random_state=42
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 1. Build preprocessing and model pipeline

Preprocessing is fitted only on training folds because it is contained within the pipeline. This reduces leakage during cross-validation and final fitting.

In [ ]:
numeric_features = ["floor_area", "rooms", "building_age"]
categorical_features = ["district", "property_type"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
    ))
])
model.fit(X_train, y_train)

## 2. Evaluate once on the held-out test set

MAE is expressed in the target's units. RMSE penalises larger errors more heavily. R² measures relative fit against a mean-only baseline and may be negative on poor predictions.

In [ ]:
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.Series({"MAE":mae, "RMSE":rmse, "R2":r2}, name="test_result")
display(metrics.to_frame().round(3))

## 3. Cross-validation on the training portion

Cross-validation provides a more stable view of model performance than a single split. The final test set is kept outside this model-selection step.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring={"mae":"neg_mean_absolute_error","r2":"r2"},
    n_jobs=-1, return_train_score=False
)
print("CV MAE mean:", round(-scores["test_mae"].mean(),3))
print("CV MAE std: ", round(scores["test_mae"].std(),3))
print("CV R2 mean: ", round(scores["test_r2"].mean(),3))

## 4. Permutation importance

Permutation importance measures how much a fitted model's score worsens when one feature is shuffled. Correlated features can share importance, so rankings should be interpreted cautiously.

In [ ]:
perm = permutation_importance(
    model, X_test, y_test, n_repeats=8, random_state=42,
    scoring="neg_mean_absolute_error", n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance.round(4))

## Next steps

- Compare the Random Forest against a dummy baseline and a regularised linear model.
- Tune hyperparameters using training-only cross-validation.
- Inspect residuals across districts and property types.
- Report data provenance, target definition, evaluation split and uncertainty.
- Do not interpret synthetic feature importance as evidence about real property markets.